In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE_URL = "https://www.bbc.com"
#TECH_URL = "https://www.bbc.com/business"
TECH_URL =str(input("Enter url to scrap the web:"))
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Get technology page
response = requests.get(TECH_URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Collect article links
links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if "/news/articles/" in href:
        links.add(urljoin(BASE_URL, href))

print(f"Found {len(links)} articles")

data = []

# Visit each article
for link in links:
    try:
        article_page = requests.get(link, headers=headers)
        article_soup = BeautifulSoup(article_page.text, "html.parser")

        title = article_soup.find("h1")
        title = title.get_text(strip=True) if title else "No title"
        time = article_soup.find("time")
        time = time.get_text(strip=True)
        paragraphs = article_soup.find_all("p")
        
        content = " ".join(
            p.get_text(strip=True)
            for p in paragraphs
        )

        data.append({
            "title": title,
            "time":time,
            "url": link,
            "content": content
        })
        
        print(title,":\n","Time:",time,"\n","link:",link,":\n", content)


    except Exception as e:
        print("Error:", e)

df = pd.DataFrame(data)
df.to_csv("bbc_technology_news.csv", index=False)
df.to_csv("bbc_technology_news.txt", index=False)

print("Saved!")

Enter url to scrap the web: https://www.bbc.com/business


Found 22 articles
New candy stores are popping up across NYC. Why? :
 Time: 1 day ago 
 link: https://www.bbc.com/news/articles/c30ylzlygngo :
 With US consumer confidence at historic lows, it's a tough time for retailers across the country. But in and around New York City one niche sector is expanding – candy stores. Mitchell Cohen, the third-generation owner of Economy Candy, on Manhattan's Lower East Side, has a theory – people will still buy candy (or sweets, as they are called in British English) – when economic times are difficult. "The dollar isn't going as far these days," he says. "Inflation, uncertainty, all that, but there's always candy." The business, the oldest sweet shop in New York, first opened its doors in 1937, towards the end of the Great Depression. Initially it was a hat and shoe repair store, with candies sold from a cart out front as an extra earning stream. But people couldn't afford to get things repaired, Cohen says. So his grandfather entirely pivoted to wha